## Parse season links (1994 — 2026)

In [30]:
from bs4 import BeautifulSoup
import json
import re

PREFIX = "https://gameshows.ru"

html_data = """
<p><span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%90%D0%B2%D1%82%D0%BE%D0%BC%D0%BE%D0%B1%D0%B8%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9_%D0%BA%D1%83%D0%B1%D0%BE%D0%BA_1994)" title="Своя игра (Автомобильный кубок 1994)">Автокубок 1994</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%90%D0%B2%D1%82%D0%BE%D0%BC%D0%BE%D0%B1%D0%B8%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9_%D0%BA%D1%83%D0%B1%D0%BE%D0%BA_1995)" title="Своя игра (Автомобильный кубок 1995)">Автокубок 1995</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%90%D0%B2%D1%82%D0%BE%D0%BC%D0%BE%D0%B1%D0%B8%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9_%D0%BA%D1%83%D0%B1%D0%BE%D0%BA_1996)" title="Своя игра (Автомобильный кубок 1996)">Автокубок 1996</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%97%D0%BE%D0%BB%D0%BE%D1%82%D0%B0%D1%8F_%D0%B4%D1%8E%D0%B6%D0%B8%D0%BD%D0%B0,_1996)" title="Своя игра (Золотая дюжина, 1996)">Золотая дюжина (1996)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%97%D0%BE%D0%BB%D0%BE%D1%82%D0%B0%D1%8F_%D0%B4%D1%8E%D0%B6%D0%B8%D0%BD%D0%B0,_1997)" title="Своя игра (Золотая дюжина, 1997)">Золотая дюжина (1997)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%97%D0%BE%D0%BB%D0%BE%D1%82%D0%B0%D1%8F_%D0%B4%D1%8E%D0%B6%D0%B8%D0%BD%D0%B0,_1998)" title="Своя игра (Золотая дюжина, 1998)">Золотая дюжина (1998)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%97%D0%BE%D0%BB%D0%BE%D1%82%D0%B0%D1%8F_%D0%B4%D1%8E%D0%B6%D0%B8%D0%BD%D0%B0,_1999)" title="Своя игра (Золотая дюжина, 1999)">Золотая дюжина (1999)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%97%D0%BE%D0%BB%D0%BE%D1%82%D0%B0%D1%8F_%D0%B4%D1%8E%D0%B6%D0%B8%D0%BD%D0%B0,_2000)" title="Своя игра (Золотая дюжина, 2000)">Золотая дюжина (2000)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9A%D1%83%D0%B1%D0%BE%D0%BA_%D0%B2%D1%8B%D0%B7%D0%BE%D0%B2%D0%B0-1)" title="Своя игра (Кубок вызова-1)">Кубок вызова-1 (2001—2002)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9A%D1%83%D0%B1%D0%BE%D0%BA_%D0%B2%D1%8B%D0%B7%D0%BE%D0%B2%D0%B0-2)" title="Своя игра (Кубок вызова-2)">Кубок вызова-2 (2002)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9A%D1%83%D0%B1%D0%BE%D0%BA_%D0%B2%D1%8B%D0%B7%D0%BE%D0%B2%D0%B0-3)" title="Своя игра (Кубок вызова-3)">Кубок вызова-3 (2002)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9A%D1%83%D0%B1%D0%BE%D0%BA_%D0%B2%D1%8B%D0%B7%D0%BE%D0%B2%D0%B0-4)" title="Своя игра (Кубок вызова-4)">Кубок вызова-4 (2003)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%90%D0%B2%D1%82%D0%BE%D0%BC%D0%BE%D0%B1%D0%B8%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9_%D0%BA%D1%83%D0%B1%D0%BE%D0%BA_2004)" title="Своя игра (Автомобильный кубок 2004)">Автокубок 2004</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%AE%D0%B1%D0%B8%D0%BB%D0%B5%D0%B9%D0%BD%D1%8B%D0%B5_%D0%B8%D0%B3%D1%80%D1%8B,_2004)" title="Своя игра (Юбилейные игры, 2004)">Юбилейные игры (2004)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9A%D0%BE%D0%BC%D0%B1%D0%B8%D0%BD%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%BD%D1%8B%D0%B9_%D0%BA%D1%83%D0%B1%D0%BE%D0%BA,_2005)" title="Своя игра (Комбинированный кубок, 2005)">Комбинированный кубок (2005)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%97%D0%BE%D0%BB%D0%BE%D1%82%D0%B0%D1%8F_%D0%B4%D1%8E%D0%B6%D0%B8%D0%BD%D0%B0,_2006)" title="Своя игра (Золотая дюжина, 2006)">Золотая дюжина (2006)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%97%D0%BE%D0%BB%D0%BE%D1%82%D0%B0%D1%8F_%D0%B4%D1%8E%D0%B6%D0%B8%D0%BD%D0%B0,_2007)" title="Своя игра (Золотая дюжина, 2007)">Золотая дюжина (2007)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%90%D0%B2%D1%82%D0%BE%D0%BC%D0%BE%D0%B1%D0%B8%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9_%D0%BA%D1%83%D0%B1%D0%BE%D0%BA_2008)" title="Своя игра (Автомобильный кубок 2008)">Автокубок 2008</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%90%D0%B2%D1%82%D0%BE%D0%BC%D0%BE%D0%B1%D0%B8%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9_%D0%BA%D1%83%D0%B1%D0%BE%D0%BA_2009)" title="Своя игра (Автомобильный кубок 2009)">Автокубок 2009</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%98%D0%B3%D1%80%D1%8B_15-%D0%BB%D0%B5%D1%82%D0%B8%D1%8F,_2009)" title="Своя игра (Игры 15-летия, 2009)">Игры 15-летия (2009)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%90%D0%B2%D1%82%D0%BE%D0%BC%D0%BE%D0%B1%D0%B8%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9_%D0%BA%D1%83%D0%B1%D0%BE%D0%BA_2010)" title="Своя игра (Автомобильный кубок 2010)">Автокубок 2010</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9A%D1%83%D0%B1%D0%BE%D0%BA_%D0%BD%D0%BE%D0%B2%D0%BE%D0%B3%D0%BE_%D0%BF%D0%BE%D0%BA%D0%BE%D0%BB%D0%B5%D0%BD%D0%B8%D1%8F,_2010)" title="Своя игра (Кубок нового поколения, 2010)">Кубок нового поколения (2010)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9A%D1%83%D0%B1%D0%BE%D0%BA_%D1%82%D1%80%D1%91%D1%85_%D0%BF%D0%BE%D0%BA%D0%BE%D0%BB%D0%B5%D0%BD%D0%B8%D0%B9,_2010)" title="Своя игра (Кубок трёх поколений, 2010)">Кубок трёх поколений (2010)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%90%D0%B2%D1%82%D0%BE%D0%BC%D0%BE%D0%B1%D0%B8%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9_%D0%BA%D1%83%D0%B1%D0%BE%D0%BA_2011)" title="Своя игра (Автомобильный кубок 2011)">Автокубок 2011</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9D%D0%BE%D0%B2%D0%BE%D0%B3%D0%BE%D0%B4%D0%BD%D0%B8%D0%B9_%D0%B1%D0%BB%D0%B8%D1%86-%D1%82%D1%83%D1%80%D0%BD%D0%B8%D1%80_2011)" title="Своя игра (Новогодний блиц-турнир 2011)">Новогодний блиц-турнир 2011</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9E%D1%82%D0%BA%D1%80%D1%8B%D1%82%D1%8B%D0%B9_%D0%BA%D0%BE%D0%BC%D0%B0%D0%BD%D0%B4%D0%BD%D1%8B%D0%B9_%D1%82%D1%83%D1%80%D0%BD%D0%B8%D1%80_2012)" title="Своя игра (Открытый командный турнир 2012)">Открытый командный турнир (2012)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%A7%D0%B5%D0%BC%D0%BF%D0%B8%D0%BE%D0%BD%D0%B0%D1%82_2012)" title="Своя игра (Чемпионат 2012)">Чемпионат 2012</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9E%D1%82%D0%BA%D1%80%D1%8B%D1%82%D0%B8%D0%B5_%D1%81%D0%B5%D0%B7%D0%BE%D0%BD%D0%B0_2013)" title="Своя игра (Открытие сезона 2013)">Кубок Открытия сезона (2013)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9E%D1%82%D0%BA%D1%80%D1%8B%D1%82%D1%8B%D0%B9_%D0%BA%D0%BE%D0%BC%D0%B0%D0%BD%D0%B4%D0%BD%D1%8B%D0%B9_%D1%82%D1%83%D1%80%D0%BD%D0%B8%D1%80_2013)" title="Своя игра (Открытый командный турнир 2013)">Открытый командный турнир (2013)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%98%D0%B3%D1%80%D1%8B_20-%D0%BB%D0%B5%D1%82%D0%B8%D1%8F,_2014)" title="Своя игра (Игры 20-летия, 2014)">Игры 20-летия (2014)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%97%D0%BE%D0%BB%D0%BE%D1%82%D0%B0%D1%8F_%D0%B4%D1%8E%D0%B6%D0%B8%D0%BD%D0%B0,_2014-2015)" title="Своя игра (Золотая дюжина, 2014-2015)">Золотая дюжина (2014—2015)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%98%D0%B3%D1%80%D0%B0_%D0%B2_%D0%BB%D0%B8%D0%BD%D0%B5%D0%B9%D0%BA%D1%83,_2015)" title="Своя игра (Игра в линейку, 2015)">Игра в линейку (2015)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%98%D0%B3%D1%80%D0%B0_%D0%B2_%D0%BB%D0%B8%D0%BD%D0%B5%D0%B9%D0%BA%D1%83,_2016)" title="Своя игра (Игра в линейку, 2016)">Игра в линейку (2016)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%98%D0%B3%D1%80%D0%B0_%D0%B2_%D0%BB%D0%B8%D0%BD%D0%B5%D0%B9%D0%BA%D1%83,_2017)" title="Своя игра (Игра в линейку, 2017)">Игра в линейку (2017)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%98%D0%B3%D1%80%D0%B0_%D0%B2_%D0%BB%D0%B8%D0%BD%D0%B5%D0%B9%D0%BA%D1%83,_2018)" title="Своя игра (Игра в линейку, 2018)">Игра в линейку (2018)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%A4%D0%B8%D0%BD%D0%B0%D0%BB%D1%8C%D0%BD%D1%8B%D0%B5_%D0%B8%D0%B3%D1%80%D1%8B_2018)" title="Своя игра (Финальные игры 2018)">Финальные игры 2018</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%98%D0%B3%D1%80%D1%8B_25-%D0%BB%D0%B5%D1%82%D0%B8%D1%8F,_2019)" title="Своя игра (Игры 25-летия, 2019)">Игры 25-летия (2019)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9A%D0%BE%D0%BC%D0%B0%D0%BD%D0%B4%D0%BD%D1%8B%D0%B9_%D0%BC%D0%B8%D0%BD%D0%B8-%D1%82%D1%83%D1%80%D0%BD%D0%B8%D1%80,_2019)" title="Своя игра (Командный мини-турнир, 2019)">Командный мини-турнир (2019)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%A2%D1%83%D1%80%D0%BD%D0%B8%D1%80_%C2%AB25%2B%C2%BB,_2019)" title="Своя игра (Турнир «25+», 2019)">Турнир «25+» (2019)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%A2%D1%83%D1%80%D0%BD%D0%B8%D1%80,_2020)" title="Своя игра (Турнир, 2020)">Турнир «Через тернии к звёздам» (2020)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%97%D0%BE%D0%BB%D0%BE%D1%82%D0%B0%D1%8F_%D0%B4%D0%B5%D0%B2%D1%8F%D1%82%D0%BA%D0%B0,_2021)" title="Своя игра (Золотая девятка, 2021)">Золотая девятка (2021)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%97%D0%BE%D0%BB%D0%BE%D1%82%D0%B0%D1%8F_%D0%B4%D0%B5%D0%B2%D1%8F%D1%82%D0%BA%D0%B0,_2022)" title="Своя игра (Золотая девятка, 2022)">Золотая девятка (2022)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%97%D0%BE%D0%BB%D0%BE%D1%82%D0%B0%D1%8F_%D1%81%D0%B5%D1%80%D0%B5%D0%B4%D0%B8%D0%BD%D0%B0,_2023)" title="Своя игра (Золотая середина, 2023)">Золотая середина (2023)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%92%D0%BE%D0%B7%D0%B2%D1%80%D0%B0%D1%89%D0%B5%D0%BD%D0%B8%D0%B5_%D0%BA_%D0%B8%D1%81%D1%82%D0%BE%D0%BA%D0%B0%D0%BC,_2024)" title="Своя игра (Возвращение к истокам, 2024)">«Возвращение к истокам» (2024)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%98%D0%B3%D1%80%D1%8B_30-%D0%BB%D0%B5%D1%82%D0%B8%D1%8F,_2024)" title="Своя игра (Игры 30-летия, 2024)">Игры 30-летия (2024)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9F%D1%80%D0%BE%D0%B4%D0%BE%D0%BB%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5_%D1%81%D0%BB%D0%B5%D0%B4%D1%83%D0%B5%D1%82%E2%80%A6,_2024)" title="Своя игра (Продолжение следует…, 2024)">«Продолжение следует…» (2024)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%A4%D0%B8%D0%BD%D0%B0%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9_%D1%82%D1%83%D1%80%D0%BD%D0%B8%D1%80,_2024)" title="Своя игра (Финальный турнир, 2024)">Финальный турнир 2024</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9A%D1%83%D0%B1%D0%BE%D0%BA_%D0%9A%D0%BE%D0%BD%D1%82%D1%83%D1%80%D0%B0,_2025)" title="Своя игра (Кубок Контура, 2025)">Кубок Контура (2025)</a> |</span>
<span style="white-space: nowrap;"><a href="/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%97%D0%BE%D0%BB%D0%BE%D1%82%D0%B0%D1%8F_%D0%B4%D0%B5%D0%B2%D1%8F%D1%82%D0%BA%D0%B0,_2026)" title="Своя игра (Золотая девятка, 2026)">Золотая девятка (2026)</a> </span>
</p>
"""

def parse_links(html):
    soup = BeautifulSoup(html, 'html.parser')
    results = []

    for a in soup.find_all('a'):
        raw_title = a.get('title', '')
        link = a.get('href', '')
        
        # 1. Clean title: remove "Своя игра" and parentheses
        clean_title = re.sub(r'^Своя игра\s*', '', raw_title)
        clean_title = clean_title.strip().strip('()')
        
        # 2. Extract Year: find 4 digits at the end of the string or inside parentheses
        year_match = re.search(r'(\d{4})', clean_title)
        year = int(year_match.group(1)) if year_match else None
        
        results.append({
            "title": clean_title,
            "year": year,
            "link": PREFIX + link
        })
        
    return results

# Execute
data = parse_links(html_data)
data[8]['year'] = 2001  # Correcting the year for "Кубок вызова-1 (2001—2002)"
data[9]['year'] = 2002  # Correcting the year for "Кубок вызова-2 (2002)"
data[10]['year'] = 2002  # Correcting the year for "Кубок вызова-3 (2002)"
data[11]['year'] = 2003  # Correcting the year for "Кубок вызова-4 (2003)"

# will overwrite seasons.json if uncommented
# with open('seasons.json', 'w', encoding='utf-8') as f:
#    json.dump(data, f, ensure_ascii=False, indent=4)

# Output as JSON
print(json.dumps(data, ensure_ascii=False, indent=4))

[
    {
        "title": "Автомобильный кубок 1994",
        "year": 1994,
        "link": "https://gameshows.ru/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%90%D0%B2%D1%82%D0%BE%D0%BC%D0%BE%D0%B1%D0%B8%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9_%D0%BA%D1%83%D0%B1%D0%BE%D0%BA_1994)"
    },
    {
        "title": "Автомобильный кубок 1995",
        "year": 1995,
        "link": "https://gameshows.ru/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%90%D0%B2%D1%82%D0%BE%D0%BC%D0%BE%D0%B1%D0%B8%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9_%D0%BA%D1%83%D0%B1%D0%BE%D0%BA_1995)"
    },
    {
        "title": "Автомобильный кубок 1996",
        "year": 1996,
        "link": "https://gameshows.ru/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%90%D0%B2%D1%82%D0%BE%D0%BC%D0%BE%D0%B1%D0%B8%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9_%D0%BA%D1%83%D0%B1%D0%BE%D0%BA_1996)"
    },
    {
        "title": "Золотая дюжина, 1996",
        "year": 1996,
        "link": "https://gameshows.ru/wiki/%D0%A1%D0%B2%

In [31]:
data[8]

{'title': 'Кубок вызова-1',
 'year': 2001,
 'link': 'https://gameshows.ru/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9A%D1%83%D0%B1%D0%BE%D0%BA_%D0%B2%D1%8B%D0%B7%D0%BE%D0%B2%D0%B0-1)'}

In [43]:
def parse_games_table(html):
    soup = BeautifulSoup(html, 'html.parser')
    results = []
    
    months = {
        'января': '01', 'февраля': '02', 'марта': '03', 'апреля': '04',
        'мая': '05', 'июня': '06', 'июля': '07', 'августа': '08',
        'сентября': '09', 'октября': '10', 'ноября': '11', 'декабря': '12'
    }

    for tr in soup.find_all('tr'):
        tds = tr.find_all('td')
        if not tds:
            continue

        idx_text = tds[0].get_text(strip=True)
        # Регулярка: обязательное число, и необязательное число в скобках
        idx_match = re.search(r'^(\d+)(?:\s*\((\d+)\))?', idx_text)
        
        if not idx_match:
            continue
            
        game_index = idx_match.group(1)
        total_index = idx_match.group(2) if idx_match.group(2) else game_index
        
        # Поиск ссылки (обычно во второй или третьей колонке, если есть rowspan)
        # Проверяем все ячейки на наличие ссылки, чтобы быть гибче
        a_tag = None
        for td in tds:
            found_a = td.find('a')
            if found_a:
                # Пропускаем "красные ссылки" и ссылки на файлы/категории
                if 'new' not in found_a.get('class', []) and '/wiki/' in found_a.get('href', ''):
                    a_tag = found_a
                    break
        
        if not a_tag:
            continue
            
        link = a_tag.get('href', '')
        
        # Парсинг даты
        date_raw = a_tag.get_text(strip=True).lower()
        date_parts = re.findall(r'(\d+)\s+([а-я]+)\s+(\d{4})', date_raw)
        
        if date_parts:
            day, month_name, year = date_parts[0]
            formatted_date = f"{day.zfill(2)}-{months.get(month_name, '00')}-{year}"
        else:
            continue 

        results.append({
            "index": int(game_index),
            "total_index": int(total_index),
            "date": formatted_date,
            "link": f"https://gameshows.ru{link}" if link.startswith('/') else link
        })

    return results

In [44]:
import requests

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

url = data[0]['link']

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    print("Success!")
except requests.exceptions.RequestException as e:
    print(f"Connection failed: {e}")

Success!


In [45]:
parse_games_table(response.text)

[{'index': 1,
  'total_index': 1,
  'date': '07-04-1994',
  'link': 'https://gameshows.ru/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9E%D0%B1%D0%B7%D0%BE%D1%80_%D0%B2%D1%8B%D0%BF%D1%83%D1%81%D0%BA%D0%B0_1994-04-07)'},
 {'index': 2,
  'total_index': 2,
  'date': '14-04-1994',
  'link': 'https://gameshows.ru/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9E%D0%B1%D0%B7%D0%BE%D1%80_%D0%B2%D1%8B%D0%BF%D1%83%D1%81%D0%BA%D0%B0_1994-04-14)'},
 {'index': 3,
  'total_index': 3,
  'date': '21-04-1994',
  'link': 'https://gameshows.ru/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9E%D0%B1%D0%B7%D0%BE%D1%80_%D0%B2%D1%8B%D0%BF%D1%83%D1%81%D0%BA%D0%B0_1994-04-21)'},
 {'index': 9,
  'total_index': 9,
  'date': '02-06-1994',
  'link': 'https://gameshows.ru/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9E%D0%B1%D0%B7%D0%BE%D1%80_%D0%B2%D1%8B%D0%BF%D1%83%D1%81%D0%BA%D0%B0_1994-06-02)'},
 {'index': 11,
  'total_index': 11,
  'date': '16-06-1994',


In [46]:
def test_links(data):
    total_links = 0
    for item in data:
        url = item['link']
        year = item['year']
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status() 
        except requests.exceptions.RequestException as e:
            print(f"Connection failed: {e}")
        else:
            game_links = parse_games_table(response.text)
            total_links += len(game_links)
            print(f"{len(game_links)} links in {item['title']}")
            for link in game_links:
                if str(year) != link['date'][-4:]:
                    print(f"Year mismatch for {item['title']}: expected {year}, got {link['date'][-4:]}")

    print(f"Total links found: {total_links}")

test_links(data)

14 links in Автомобильный кубок 1994
28 links in Автомобильный кубок 1995
17 links in Автомобильный кубок 1996
7 links in Золотая дюжина, 1996
Year mismatch for Золотая дюжина, 1996: expected 1996, got 1997
49 links in Золотая дюжина, 1997
52 links in Золотая дюжина, 1998
48 links in Золотая дюжина, 1999
45 links in Золотая дюжина, 2000
38 links in Кубок вызова-1
Year mismatch for Кубок вызова-1: expected 2001, got 2002
Year mismatch for Кубок вызова-1: expected 2001, got 2002
Year mismatch for Кубок вызова-1: expected 2001, got 2002
Year mismatch for Кубок вызова-1: expected 2001, got 2002
Year mismatch for Кубок вызова-1: expected 2001, got 2002
Year mismatch for Кубок вызова-1: expected 2001, got 2002
Year mismatch for Кубок вызова-1: expected 2001, got 2002
Year mismatch for Кубок вызова-1: expected 2001, got 2002
Year mismatch for Кубок вызова-1: expected 2001, got 2002
Year mismatch for Кубок вызова-1: expected 2001, got 2002
29 links in Кубок вызова-2
48 links in Кубок вызова-3


In [ ]:
import re
import json
from bs4 import BeautifulSoup

def parse_svoya_igra(html_content, year=1994):
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Удаляем сноски [1], [2] и т.д.
    for sup in soup.find_all('sup'):
        sup.decompose()
    
    questions = []
    current_round = 0
    
    # Собираем все значимые элементы
    elements = soup.find_all(['h3', 'h4', 'p'])
    
    i = 0
    while i < len(elements):
        el = elements[i]
        
        # 1. Определение раунда
        if el.name == 'h3':
            text = el.get_text().strip()
            if "Синий раунд" in text: current_round = 1
            elif "Красный раунд" in text: current_round = 2
            elif "Финальный раунд" in text: current_round = "Final"
            i += 1
            continue

        # 2. Обработка вопроса (тема и цена)
        if el.name == 'h4' and current_round != "Final":
            title = el.get_text().strip()
            match = re.search(r'^(.*?)\s*\((\d+)\)', title)
            if match:
                topic = match.group(1)
                price = int(match.group(2))
                
                question_parts = []
                correct_answer = ""
                players_answers = []
                
                # Собираем содержимое после заголовка h4
                j = i + 1
                while j < len(elements) and elements[j].name == 'p':
                    p_tag = elements[j]
                    raw_text = p_tag.get_text().strip()
                    
                    # Проверяем, это лог (ответы) или еще текст вопроса
                    if any(x in raw_text for x in ["Отвечает", "Играет", "Правильный ответ:"]):
                        # Разбиваем абзац на строки, сохраняя структуру <br/>
                        lines = [line.strip() for line in p_tag.get_text(separator="\n").split("\n") if line.strip()]
                        
                        was_player_called = False # Флаг: была ли строка "Отвечает ..."
                        
                        for line in lines:
                            # 1. Если строка про то, кто отвечает
                            if line.startswith("Отвечает") or line.startswith("Играет"):
                                was_player_called = True
                                continue
                            
                            # 2. Если это неправильный ответ игрока
                            if "Ответ игрока:" in line:
                                player_val = line.split(":", 1)[1].strip()
                                players_answers.append(player_val)
                                was_player_called = False # Сбрасываем флаг
                            
                            # 3. Если это правильный ответ
                            elif "Правильный ответ:" in line:
                                print(line)
                                correct_val = line.split(":", 1)[1].strip()
                                correct_answer = correct_val
                                
                                # ЛОГИКА: если перед этим был вызов игрока и НЕ БЫЛО строки "Ответ игрока",
                                # значит игрок сразу ответил правильно.
                                if was_player_called:
                                    players_answers.append(correct_val)
                                
                                was_player_called = False
                            
                            # Игнорируем ставки и прочее
                            elif "Ставка" in line:
                                pass
                            else:
                                was_player_called = False

                        # Если нашли правильный ответ, то вопрос закончен
                        if correct_answer:
                            j += 1
                            break
                    else:
                        # Накопление текста вопроса (игнорируем список тем в начале раунда)
                        if "Темы:" not in raw_text:
                            question_parts.append(raw_text)
                    j += 1
                
                questions.append({
                    "year": year,
                    "price": price,
                    "round": current_round,
                    "topic": topic,
                    "question": " ".join(question_parts).strip(),
                    "answer": correct_answer,
                    "players_answers": players_answers
                })
                i = j - 1
        
        # 3. Финальный раунд
        elif current_round == "Final" and el.name == 'p' and "Тема:" in el.get_text():
            topic = el.get_text().replace("Тема:", "").strip()
            i += 1
            question_text = elements[i].get_text(strip=True) if i < len(elements) else ""
            
            correct_answer = ""
            players_answers = []
            
            j = i + 1
            while j < len(elements) and current_round == "Final":
                p_tag = elements[j]
                p_text = p_tag.get_text().strip()
                
                if "Правильный ответ:" in p_text:
                    correct_answer = p_text.split(":", 1)[1].strip()
                    break
                elif "Ответ" in p_text and ":" in p_text:
                    # В финале формат: "Ответ Имя: Текст [Ставка]"
                    # Берем часть после имени до слова Ставка
                    ans_part = p_text.split(":", 1)[1]
                    ans_clean = ans_part.split("Ставка")[0].strip()
                    players_answers.append(ans_clean)
                j += 1
            
            questions.append({
                "year": year,
                "price": 0,
                "round": "Final",
                "topic": topic,
                "question": question_text,
                "answer": correct_answer,
                "players_answers": players_answers
            })
            break
            
        i += 1
    return questions

In [ ]:
import requests

url = "https://gameshows.ru/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9E%D0%B1%D0%B7%D0%BE%D1%80_%D0%B2%D1%8B%D0%BF%D1%83%D1%81%D0%BA%D0%B0_1994-04-07)"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
    'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7',
}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status() # Выдаст ошибку, если статус не 200
    response.encoding = 'utf-8'  # Явно указываем кодировку для кириллицы
    
    html_content = response.text
    print("Успешно загружено!")
    
except requests.exceptions.RequestException as e:
    print(f"Ошибка при загрузке: {e}")

Успешно загружено!


In [15]:
json_data = parse_svoya_igra(html_content, 1994)
print(json.dumps(json_data, ensure_ascii=False, indent=4))

Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
[
    {
        "year": 1994,
        "price": 30,
        "round": 1,
        "topic": "В мире животных",
        "question": "В Англии

In [19]:
import re
import json
import requests
from bs4 import BeautifulSoup

def parse_svoya_igra_fixed(url, year=1994):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.encoding = 'utf-8'
        if response.status_code != 200:
            print(f"Ошибка доступа: {response.status_code}")
            return []
        html_content = response.text
    except Exception as e:
        print(f"Ошибка сети: {e}")
        return []

    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Очистка от мусора
    for sup in soup.find_all('sup'): sup.decompose()
    
    content = soup.find('div', class_='mw-parser-output')
    if not content:
        print("Контент не найден")
        return []

    questions = []
    current_round = 0
    
    # Ищем ВСЕ заголовки и абзацы внутри контента в порядке их появления
    # Это решает проблему вложенности в div.mw-heading
    all_elements = content.find_all(['h3', 'h4', 'p'])
    
    i = 0
    while i < len(all_elements):
        el = all_elements[i]
        # Заменяем неразрывные пробелы \xa0 на обычные
        text = el.get_text().replace('\xa0', ' ').strip()

        # 1. Определение раунда
        if el.name == 'h3':
            if "Синий раунд" in text: current_round = 1
            elif "Красный раунд" in text: current_round = 2
            elif "Финальный раунд" in text: current_round = "Final"
            i += 1
            continue

        # 2. Обычные вопросы
        if el.name == 'h4' and current_round != "Final":
            # Регулярка для темы и цены
            match = re.search(r'(.+?)\s*\((\d+)\)', text)
            if match:
                topic = match.group(1).strip()
                price = int(match.group(2))
                
                q_text = ""
                correct_ans = ""
                p_answers = []
                
                # Собираем все <p> после h4 до следующего заголовка
                j = i + 1
                while j < len(all_elements) and all_elements[j].name == 'p':
                    p_tag = all_elements[j]
                    p_text = p_tag.get_text(" ", strip=True).replace('\xa0', ' ')
                    
                    if any(x in p_text for x in ["Отвечает", "Играет", "Правильный ответ"]):
                        # Логика извлечения ответов через регулярное разделение
                        # Мы ищем все вхождения меток и текст после них
                        parts = re.split(r'(Отвечает|Играет|Ответ игрока:|Правильный ответ:)', p_text)
                        
                        was_called = False
                        # В списке parts: [текст_до, метка, текст_после, метка, текст_после...]
                        for k in range(1, len(parts), 2):
                            label = parts[k]
                            val = parts[k+1].strip().rstrip('.').strip()
                            
                            if label in ["Отвечает", "Играет"]:
                                was_called = True
                            elif label == "Ответ игрока:":
                                p_answers.append(val)
                                was_called = False
                            elif label == "Правильный ответ:":
                                correct_ans = val
                                if was_called:
                                    p_answers.append(val)
                                was_called = False
                    else:
                        # Накопление текста вопроса
                        if "Темы:" not in p_text and p_text:
                            q_text += " " + p_text
                    j += 1
                
                questions.append({
                    "year": year,
                    "price": price,
                    "round": current_round,
                    "topic": topic,
                    "question": q_text.strip(),
                    "answer": correct_ans,
                    "players_answers": p_answers
                })
                i = j - 1 # Перескакиваем обработанные абзацы
        
        # 3. Финальный раунд
        elif current_round == "Final" and el.name == 'p' and "Тема:" in text:
            topic = text.replace("Тема:", "").strip()
            i += 1
            # Следующий p - вопрос
            q_text = all_elements[i].get_text(strip=True) if i < len(all_elements) else ""
            
            ans = ""
            p_ans = []
            
            j = i + 1
            while j < len(all_elements) and all_elements[j].name == 'p':
                p_text = all_elements[j].get_text(" ", strip=True).replace('\xa0', ' ')
                if "Правильный ответ:" in p_text:
                    ans = p_text.split(":", 1)[1].strip()
                    break
                elif "Ответ" in p_text and ":" in p_text:
                    # Извлекаем текст до слова "Ставка"
                    val = p_text.split(":", 1)[1].split("Ставка")[0].strip()
                    p_ans.append(val)
                j += 1
            
            questions.append({
                "year": year,
                "price": 0,
                "round": "Final",
                "topic": topic,
                "question": q_text,
                "answer": ans,
                "players_answers": p_ans
            })
            break # Финал обычно в конце
            
        i += 1
    return questions

# Проверка
url = "https://gameshows.ru/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9E%D0%B1%D0%B7%D0%BE%D1%80_%D0%B2%D1%8B%D0%BF%D1%83%D1%81%D0%BA%D0%B0_1994-04-07)"
data = parse_svoya_igra_fixed(url, 1994)

if data:
    print(f"Questions parsed: {len(data)}")
    print(json.dumps(data, ensure_ascii=False, indent=4))
else:
    print("Questions list is empty. Check the connection or page structure.")

Questions parsed: 49
[
    {
        "year": 1994,
        "price": 30,
        "round": 1,
        "topic": "В мире животных",
        "question": "В Англии бетонные бункеры, оставшиеся после Второй мировой войны, начали переделывать в искусственные пещеры. Они предназначались для зимовки…",
        "answer": "Летучих мышей",
        "players_answers": [
            "Пчёл"
        ]
    },
    {
        "year": 1994,
        "price": 20,
        "round": 1,
        "topic": "В мире животных",
        "question": "Учёные считали, что эта рыба вымерла 70 млн лет назад, а она преспокойно плавает у южных берегов Африки.",
        "answer": "Латимерия",
        "players_answers": [
            "Латимерия"
        ]
    },
    {
        "year": 1994,
        "price": 20,
        "round": 1,
        "topic": "Книжный мир",
        "question": "Он писал: «Я — москвич! Сколь счастлив тот, кто может произносить это слово, вкладывая в него всего себя. Я — москвич!»",
        "answer": "Владимир 